# Building merges at stride 8

# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from tqdm.auto import tqdm

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.segmentation import CLASSES, LRASPP_OUTPUT_STRIDE_PX
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.eval import (bilinear_argmax, label_at_stride, load_baseline_model, load_val_sets,
                                      run_model)
from uavloc.track import REPO_ROOT

# Params

In [ ]:
DEVICE = "cpu"
M_PER_PX = 0.385
MIN_BUILDING_PX = 16
MATCH_FRAC = 0.2
CONNECTIVITY = 1
GAP_MARGIN_PX = 40
GAP_BINS_M = [0, 1, 2, 3, 4, 6, 8, 12, 16]
EXAMPLE = ["bushcha", 0, 1664, 1472]
ZOOM_PX = 256
ORACLE_STRIDES = [8, 4]
FIGURES = "pocs/building-merge-stride/figures"

In [ ]:
BUILDING = CLASSES.index("building")
CELL_M = LRASPP_OUTPUT_STRIDE_PX * M_PER_PX
CELL_M_BY_STRIDE = {stride: stride * M_PER_PX for stride in ORACLE_STRIDES}
STRUCTURE = ndi.generate_binary_structure(2, CONNECTIVITY)
STATUSES = ["separate", "merged", "missed"]
STATUS_RGB = {"separate": (40, 170, 60), "merged": (240, 150, 0), "missed": (220, 40, 40)}
CELL_M

# Data

In [ ]:
val_sets = load_val_sets()
{name: [a.rgb.shape[:2] for a in images] for name, images in val_sets.items()}

# Model

In [ ]:
segmenter = load_baseline_model(DEVICE)

In [ ]:
cases = run_model(segmenter, val_sets, DEVICE)

# Run

In [ ]:
def building_maps(case) -> dict[str, np.ndarray]:
    """Boolean building masks: the label, the perfect prediction at each oracle stride (nearest / bilinear), LR-ASPP."""
    label = case.image.label
    valid = label != IGNORE_LABEL
    maps = {"label": label == BUILDING}
    for stride in ORACLE_STRIDES:
        for mode in ("nearest", "bilinear"):
            maps[f"stride-{stride} oracle, {mode}"] = (label_at_stride(label, mode, stride) == BUILDING) & valid
    maps["LR-ASPP"] = (bilinear_argmax(case.output, case.image.rgb) == BUILDING) & valid
    return maps

In [ ]:
def components(mask: np.ndarray) -> tuple[np.ndarray, int]:
    """Connected components with fewer than `MIN_BUILDING_PX` pixels removed, relabelled 1..n."""
    lab, n = ndi.label(mask, STRUCTURE)
    keep = np.bincount(lab.ravel(), minlength=n + 1) >= MIN_BUILDING_PX
    keep[0] = False
    relabel = np.zeros(n + 1, np.int32)
    relabel[keep] = np.arange(1, keep.sum() + 1)
    return relabel[lab], int(keep.sum())

In [ ]:
def overlap_fractions(gt: np.ndarray, n_gt: int, pred: np.ndarray, n_pred: int) -> np.ndarray:
    """`(n_gt, n_pred)`: share of each labelled building's area covered by each predicted component."""
    both = (gt > 0) & (pred > 0)
    pairs = np.bincount((gt[both] - 1) * n_pred + (pred[both] - 1), minlength=n_gt * n_pred)
    area = np.bincount(gt.ravel(), minlength=n_gt + 1)[1:]
    return pairs.reshape(n_gt, n_pred) / np.maximum(area, 1)[:, None]

In [ ]:
def building_status(frac: np.ndarray) -> list[str]:
    """Per labelled building: missed (no component covers >= MATCH_FRAC of it), merged (a component that covers it
    also covers another building) or separate."""
    hit = frac >= MATCH_FRAC
    shared = hit.sum(0) >= 2
    return ["missed" if not row.any() else "merged" if shared[row].any() else "separate" for row in hit]

In [ ]:
def nearest_gap_px(gt: np.ndarray, n_gt: int) -> np.ndarray:
    """Per labelled building: distance to the nearest other building, capped at `GAP_MARGIN_PX`."""
    gaps = np.full(n_gt, float(GAP_MARGIN_PX))
    for i, box in enumerate(ndi.find_objects(gt)):
        rows = slice(max(box[0].start - GAP_MARGIN_PX, 0), box[0].stop + GAP_MARGIN_PX)
        cols = slice(max(box[1].start - GAP_MARGIN_PX, 0), box[1].stop + GAP_MARGIN_PX)
        crop = gt[rows, cols]
        other = (crop > 0) & (crop != i + 1)
        if other.any():
            gaps[i] = min(ndi.distance_transform_edt(~other)[crop == i + 1].min(), GAP_MARGIN_PX)
    return gaps

In [ ]:
def image_table(name: str, case) -> pd.DataFrame:
    """One row per labelled building of one val image: area, gap to its neighbour, status under each map."""
    maps = building_maps(case)
    gt, n_gt = components(maps.pop("label"))
    table = pd.DataFrame({"set": name, "date": case.image.date, "building": np.arange(1, n_gt + 1),
                          "area_m2": np.bincount(gt.ravel(), minlength=n_gt + 1)[1:] * M_PER_PX ** 2,
                          "gap_m": nearest_gap_px(gt, n_gt) * M_PER_PX})
    for key, mask in maps.items():
        pred, n_pred = components(mask)
        table[key] = building_status(overlap_fractions(gt, n_gt, pred, n_pred))
    return table

In [ ]:
jobs = [(name, case) for name, items in cases.items() for case in items]
buildings = pd.concat([image_table(name, case) for name, case in tqdm(jobs, desc="images")], ignore_index=True)
MAPS = [c for c in buildings.columns if c not in ("set", "date", "building", "area_m2", "gap_m")]
buildings

In [ ]:
buildings.groupby("set")[["area_m2", "gap_m"]].describe().round(1).T

In [ ]:
def status_shares(frame: pd.DataFrame) -> pd.DataFrame:
    """Share of labelled buildings that are separate / merged / missed, per map (rows) and status (columns)."""
    return pd.DataFrame({m: frame[m].value_counts(normalize=True) for m in MAPS}).T.reindex(columns=STATUSES).fillna(0)

In [ ]:
shares = pd.concat({name: status_shares(frame) for name, frame in [("all", buildings), *buildings.groupby("set")]})
shares.round(3)

In [ ]:
counts = buildings.groupby("set").size().rename("labelled buildings").to_frame()
for stride, cell_m in CELL_M_BY_STRIDE.items():
    counts[f"gap < 1 stride-{stride} cell"] = buildings[buildings.gap_m < cell_m].groupby("set").size()
counts

In [ ]:
def merged_by_gap(frame: pd.DataFrame) -> pd.DataFrame:
    """Share of buildings merged under each map, by the gap to their nearest neighbour (bins in metres)."""
    gap_bin = pd.cut(frame.gap_m, GAP_BINS_M + [np.inf], right=False)
    out = pd.DataFrame({m: (frame[m] == "merged").groupby(gap_bin, observed=False).mean() for m in MAPS})
    out["buildings"] = frame.groupby(gap_bin, observed=False).size()
    return out

In [ ]:
by_gap = merged_by_gap(buildings)
by_gap.round(3)

In [ ]:
def lost_share(frame: pd.DataFrame) -> pd.DataFrame:
    """Of the buildings LR-ASPP merges: what the perfect map at each stride (bilinear) does with them. Merged there
    = lost to that resolution; separate there = a model error at that resolution."""
    merged = frame[frame["LR-ASPP"] == "merged"]
    return pd.DataFrame({f"stride-{s} oracle": merged[f"stride-{s} oracle, bilinear"].value_counts()
                         for s in ORACLE_STRIDES}).reindex(STATUSES).fillna(0).astype(int)

In [ ]:
lost_share(buildings)

# Plots

In [ ]:
fig_status, axes = plt.subplots(1, len(MAPS), figsize=(4 * len(MAPS), 3.8), sharey=True)
for ax, m in zip(axes, MAPS):
    shares.xs(m, level=1).plot.bar(stacked=True, ax=ax, color=[np.array(STATUS_RGB[s]) / 255 for s in STATUSES],
                                    legend=False)
    ax.set_title(m); ax.tick_params(axis="x", labelrotation=20)
axes[0].set_ylabel("share of labelled buildings"); axes[-1].legend(STATUSES, fontsize=8)
fig_status.tight_layout()

In [ ]:
fig_gap, ax = plt.subplots(figsize=(7, 4))
centres = [(a + b) / 2 for a, b in zip(GAP_BINS_M, GAP_BINS_M[1:] + [GAP_BINS_M[-1] + 4])]
for m in MAPS:
    ax.plot(centres, by_gap[m].values, "-o", label=m)
for (stride, cell_m), ls in zip(CELL_M_BY_STRIDE.items(), ("--", ":")):
    ax.axvline(cell_m, color="k", ls=ls, lw=.8, label=f"one stride-{stride} cell ({cell_m:.2f} m)")
ax.set_xlabel("gap to nearest labelled building (m)"); ax.set_ylabel("share merged")
ax.grid(alpha=.3); ax.legend(fontsize=8)
fig_gap.tight_layout()

In [ ]:
def status_overlay(gt: np.ndarray, statuses: list[str], pred: np.ndarray) -> np.ndarray:
    """Labelled buildings coloured by status, predicted components' outlines in black."""
    out = np.full((*gt.shape, 3), 255, np.uint8)
    colours = np.array([(255, 255, 255)] + [STATUS_RGB[s] for s in statuses], np.uint8)
    out[gt > 0] = colours[gt[gt > 0]]
    edge = (pred > 0) & ~ndi.binary_erosion(pred > 0)
    out[edge] = 0
    return out

In [ ]:
def example_panels(name: str, index: int, top: int, left: int) -> list[tuple[np.ndarray, str]]:
    """RGB, then per map: status of every labelled building in the window plus that map's outlines."""
    case, win = cases[name][index], np.s_[top:top + ZOOM_PX, left:left + ZOOM_PX]
    maps = building_maps(case)
    gt, n_gt = components(maps.pop("label"))
    panels = [(case.image.rgb[win], f"{name} {case.image.date}")]
    for key, mask in maps.items():
        pred, n_pred = components(mask)
        panels.append((status_overlay(gt, building_status(overlap_fractions(gt, n_gt, pred, n_pred)), pred)[win], key))
    return panels

In [ ]:
fig_example, axes = plt.subplots(1, 1 + len(MAPS), figsize=(3.6 * (1 + len(MAPS)), 4.2))
for ax, (img, title) in zip(axes, example_panels(*EXAMPLE)):
    ax.imshow(img, interpolation="nearest"); ax.set_title(title, fontsize=9); ax.axis("off")
fig_example.legend([plt.Rectangle((0, 0), 1, 1, color=np.array(STATUS_RGB[s]) / 255) for s in STATUSES], STATUSES,
                   loc="lower center", ncol=3)
fig_example.tight_layout(rect=(0, .06, 1, 1))

In [ ]:
for fig, stem in [(fig_status, "status-shares"), (fig_gap, "merged-by-gap"), (fig_example, "example-bushcha")]:
    fig.savefig(REPO_ROOT / FIGURES / f"{stem}.png", dpi=120, bbox_inches="tight")